# 14장 코딩 에이전트와 멀티 에이전트

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `14_코딩_에이전트와_멀티_에이전트.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install transformers
# 예제 14-2는 1장의 토크나이저만 씁니다(모델은 내려받지 않습니다)
import shutil
import subprocess
import tempfile
from pathlib import Path

import embed
LOCAL_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
embed.ask_endpoint(LOCAL_MODEL)          # 토크나이저가 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)

MATERIAL = Path(DATA) / '14장'           # 가상의 설비팀 작업 폴더 재료


def front_matter(path):
    """SKILL.md 맨 앞의 --- 두 줄 사이(front matter)를 읽어 {'name': ..., 'description': ...}로 돌려줍니다"""
    head = path.read_text(encoding='utf-8').split('---')[1]
    return dict(line.split(': ', 1) for line in head.strip().splitlines())


def make_project():
    """임시 폴더에 가상의 설비팀 작업 폴더를 만들고 경로를 돌려줍니다.

    규칙파일.md는 CLAUDE.md로, skills/는 .claude/skills/로, 기록/은 그대로 복사합니다.
    """
    project = Path(tempfile.mkdtemp(prefix='work14_'))
    shutil.copy(MATERIAL / '규칙파일.md', project / 'CLAUDE.md')
    shutil.copytree(MATERIAL / 'skills', project / '.claude' / 'skills')
    shutil.copytree(MATERIAL / '기록', project / '기록')
    return project


print('토크나이저:', LOCAL_MODEL, '/ 재료:', MATERIAL)


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 14-1. 질문 한 줄의 비용

교재의 절: "코딩 에이전트"

임시 폴더에 가상의 설비팀 작업 폴더(규칙 파일, Skill 세 개, 기록 파일)를 만들고, 그 폴더에서 Claude Code를 헤드리스 모드로 한 번 실행합니다. 에이전트 턴 수, 보낸 입력 토큰의 합과 그중 캐시에서 읽은 토큰, 응답이 규칙 파일의 "출처 파일 이름을 괄호로 붙인다" 규칙을 지켰는지 봅니다. Claude Code의 소형 모델로 실행하며 요금이 듭니다.

In [ ]:
if shutil.which('claude') is None:
    print('Claude Code가 설치되어 있지 않아 이 예제를 실행하지 않습니다.')
else:
    project = make_project()        # 준비 셀의 함수: 임시 폴더에 CLAUDE.md, .claude/skills/, 기록/을 만듭니다
    command = ['claude', '-p', 'W-07의 캐필러리 교체 기준과 마지막 교체일을 알려 주세요.',
               '--output-format', 'json',          # 응답과 사용량을 JSON 하나로 받습니다
               '--model', 'haiku',                 # Claude Code의 모델 별칭. 소형 모델입니다
               '--permission-mode', 'dontAsk']     # 물어야 하는 도구 실행은 모두 거부합니다(읽기는 묻지 않음)
    done = subprocess.run(command, cwd=project, capture_output=True, text=True, encoding='utf-8')
    run = json.loads(done.stdout)
    usage = run['usage']                # 이번 실행의 모든 모델 요청을 합친 사용량
    sent = usage['input_tokens'] + usage['cache_read_input_tokens'] + usage['cache_creation_input_tokens']
    print(f"에이전트 턴 {run['num_turns']}회, 보낸 입력 {sent:,}토큰(캐시 읽기 {usage['cache_read_input_tokens']:,}), "
          f"출력 {usage['output_tokens']:,}토큰, {run['total_cost_usd']:.4f}달러")
    print(run['result'])

에이전트 턴 5회, 보낸 입력 108,167토큰(캐시 읽기 86,227), 출력 1,176토큰, 0.0042달러
**W-07 캐필러리 교체 기준과 마지막 교체일**

- **교체 기준:** 본딩 80만 회마다 교체함(점검기준.md).
- **마지막 교체일:** 2026-11-18임(점검기록.csv). 교체 시점 누적은 본딩 71만 회였음(점검기록.csv). 알람로그.csv에도 2026-11-18 02:37에 "부품 교체: 캐필러리" 기록이 있음.
- **교체 전 징후:** 2026-11-17 PM 점검에서 캐필러리 끝 마모가 관찰됨(점검기록.csv).
- **현재 누적 본딩 횟수:** 기록 없음. 교체 후 본딩 횟수를 기록한 파일이 없어 다음 교체 예정 시점은 계산하지 않음.


## 예제 14-2. Skill의 세 단계 토큰 수

교재의 절: "Skill과 점진적 공개"

재료의 규칙 파일과 Skill 세 개를 1장의 토크나이저로 셉니다. 규칙 파일은 모든 요청에 전체가 들어가고, Skill은 1단계(목록의 한 줄)만 늘 들어갑니다. 1단계와 2단계, 3단계의 토큰 수가 몇 배 차이 나는지 봅니다.

In [ ]:
def tokens(text):
    return len(tokenizer.encode(text))


rules = (MATERIAL / '규칙파일.md').read_text(encoding='utf-8')
print(f'규칙 파일: {tokens(rules):,}토큰. 세션의 모든 요청에 들어갑니다\n')
for folder in sorted((MATERIAL / 'skills').iterdir()):
    meta = front_matter(folder / 'SKILL.md')                       # 준비 셀의 함수: name, description
    listing = f"- {meta['name']}: {meta['description']}"          # 1단계: Skill 목록의 한 줄
    body = (folder / 'SKILL.md').read_text(encoding='utf-8')       # 2단계: SKILL.md 전체
    files = sum(tokens(path.read_text(encoding='utf-8'))           # 3단계: 딸린 파일
                for path in folder.glob('*.md') if path.name != 'SKILL.md')
    print(f'{folder.name}\n  1단계 {tokens(listing):,}토큰, 2단계 {tokens(body):,}토큰, 3단계 {files:,}토큰')

규칙 파일: 395토큰. 세션의 모든 요청에 들어갑니다

checking-safety-notice
  1단계 74토큰, 2단계 412토큰, 3단계 261토큰
summarizing-shift-handover
  1단계 68토큰, 2단계 288토큰, 3단계 0토큰
writing-alarm-report
  1단계 80토큰, 2단계 582토큰, 3단계 391토큰


## 예제 14-3. 설명만 보고 Skill 고르기

교재의 절: "Skill과 점진적 공개"

Claude Code가 Skill 목록을 시스템 프롬프트에 넣는 방식을 단순하게 흉내 내, 모델에게 요청마다 읽을 Skill을 고르게 합니다. 같은 요청 네 개를 이름과 설명이 모호한 목록과, 무엇과 언제를 적은 재료의 목록에 각각 보냅니다. 마지막 요청은 어느 Skill도 필요 없는 일반 질문입니다. 이 셀은 API를 8번 호출합니다.

In [ ]:
vague = {'helper': '업무를 돕는 양식 모음입니다.', 'checker': '문서를 확인합니다.', 'notes': '메모를 다룹니다.'}
clear = {meta['name']: meta['description']
         for meta in (front_matter(path) for path in sorted((MATERIAL / 'skills').glob('*/SKILL.md')))}
requests = ['W-07의 이번 주 알람 현황을 정리해 주세요.',
            '내일 게시할 안전 공지 초안인데 규정에 맞는지 봐 주세요.',
            '야간조가 남긴 인수인계 메모를 표로 바꿔 주세요.',
            '와이어 본딩에서 캐필러리는 어떤 역할을 하나요?']
for label, skills in [('모호한 설명', vague), ('무엇과 언제를 적은 설명', clear)]:
    listing = '\n'.join(f'- {name}: {description}' for name, description in skills.items())
    system = (f'너는 코딩 에이전트다. 쓸 수 있는 Skill은 다음과 같다.\n{listing}\n'
              '요청을 처리하기 전에 읽을 Skill이 있으면 그 이름만, 없으면 "없음"이라고만 답한다.')
    print(f'[{label}]')
    for request in requests:
        reply = llm.ask(request, system=system, model='소형', max_tokens=1000)   # thinking에도 출력 토큰을 쓰므로 넉넉히 둡니다
        print(f'  {request[:22]}… → {reply.text.strip()}')

[모호한 설명]
  W-07의 이번 주 알람 현황을 정리해 … → 없음
  내일 게시할 안전 공지 초안인데 규정에 … → checker
  야간조가 남긴 인수인계 메모를 표로 바꿔… → notes
  와이어 본딩에서 캐필러리는 어떤 역할을 … → 없음
[무엇과 언제를 적은 설명]
  W-07의 이번 주 알람 현황을 정리해 … → writing-alarm-report
  내일 게시할 안전 공지 초안인데 규정에 … → checking-safety-notice
  야간조가 남긴 인수인계 메모를 표로 바꿔… → summarizing-shift-handover
  와이어 본딩에서 캐필러리는 어떤 역할을 … → 없음


## 예제 14-4. 서브에이전트가 메인에 남기는 것

교재의 절: "서브에이전트와 멀티 에이전트"

알람 로그 400행에서 부품 교체 기록을 찾는 일을 두 방법으로 처리하고, 메인이 이어서 질문할 때 그 요청의 입력 토큰을 비교합니다. 직접 읽기는 메인의 대화에 로그가 남고, 서브에이전트는 따로 보낸 요청이 로그를 읽고 메인에는 요약만 돌아옵니다. 서브에이전트를 `llm.ask()` 요청 하나로 단순하게 흉내 낸 것입니다. 이 셀은 API를 4번 호출합니다.

In [ ]:
log = (MATERIAL / '기록' / '알람로그.csv').read_text(encoding='utf-8')
task = '알람 로그에서 조치가 "부품 교체"로 시작하는 행을 모두 찾아 일시, 설비, 부품을 알려 주세요.'
follow = '그 가운데 와이어 본더에 대해, 다음 PM 때 확인할 것을 한 문장으로 제안해 주세요.'
# 직접 읽기: 로그 전체가 메인의 대화에 남습니다
direct = llm.ask(f'{task}\n\n{log}', model='소형', max_tokens=500)
main_direct = [{'role': 'user', 'content': f'{task}\n\n{log}'}, {'role': 'assistant', 'content': direct.text}]
# 서브에이전트에 맡기기: 따로 보낸 요청이 로그를 읽고, 메인에는 위임 메시지와 요약만 남습니다
sub = llm.ask(f'{task} 결과만 세 줄 이내로 답해 주세요.\n\n{log}', model='소형', max_tokens=500)
main_sub = [{'role': 'user', 'content': task}, {'role': 'assistant', 'content': sub.text}]
print(f'서브에이전트: 입력 {sub.input_tokens:,}토큰, 출력 {sub.output_tokens:,}토큰(thinking 포함). 메인에 돌려준 요약:\n{sub.text.strip()}\n')
for label, history in [('직접 읽기', main_direct), ('서브에이전트', main_sub)]:
    reply = llm.ask(follow, history=history, model='소형', max_tokens=500)
    print(f'[{label}] 메인의 다음 요청 입력 {reply.input_tokens:,}토큰 → {reply.text.strip()}')

서브에이전트: 입력 15,418토큰, 출력 290토큰(thinking 포함). 메인에 돌려준 요약:
2026-11-18 02:37, W-07, 캐필러리
2026-11-20 00:51, B-04, 흡착 노즐
2026-11-21 10:13, M-03, 히터 카트리지

[직접 읽기] 메인의 다음 요청 입력 15,589토큰 → 다음 PM에서 W-07의 캐필러리 장착 상태와 마모를 점검하고, 교체 후에도 와이어 끊김(E-214)이 계속 발생하므로 와이어 텐션과
[서브에이전트] 메인의 다음 요청 입력 184토큰 → 다음 PM 때 W-07의 캐필러리 장착 상태와 팁 마모를 점검하고, 교체 후 첫 본딩 샘플의 접합 강도와 와이어 루프 형상을 확인하세요.

(캐필러리는 와이어 본더의 소모품이므로 W-07을 와이어 본더로 보고 답했습니다.)


## 실습 14-1. 차단 훅 작성

교재의 절: "개념 확인 실습"

`PreToolUse` 훅의 판단 부분 `guard(event)`를 작성합니다. `event`는 Claude Code가 훅에 넘기는 JSON을 dict로 읽은 것입니다. 파일 도구(Edit, Write)로 `기록/`의 파일을 바꾸는 요청과, `기록`이 나오면서 `rm `, `del `, `remove`, `>`, `mv ` 중 하나가 든 셸 명령을 막습니다. 도구 요청 일곱 건마다 종료 코드가 0일지 2일지 먼저 `my_guess`에 적고, 막지 못한 요청이 있다면 그 이유를 한 문장으로 설명하면 끝납니다.

In [ ]:
def guard(event):
    """PreToolUse 훅의 판단 부분. 막으면 (2, 이유)를, 통과시키면 (0, '')을 돌려줍니다"""
    return 0, ''  # TODO


events = [
    {'tool_name': 'Read', 'tool_input': {'file_path': '기록/알람로그.csv'}},
    {'tool_name': 'Edit', 'tool_input': {'file_path': '기록/점검기준.md'}},
    {'tool_name': 'Write', 'tool_input': {'file_path': '보고서/W-07_2026-W47_알람.md'}},
    {'tool_name': 'Bash', 'tool_input': {'command': 'rm 기록/알람로그.csv'}},
    {'tool_name': 'Bash', 'tool_input': {'command': "python -c \"import os; os.remove('기록/알람로그.csv')\""}},
    {'tool_name': 'Bash', 'tool_input': {'command': 'python tools/정리.py'}},  # 스크립트가 안에서 기록/의 파일을 지운다고 가정
    {'tool_name': 'Bash', 'tool_input': {'command': 'wc -l 기록/알람로그.csv'}},
]
my_guess = [None] * 7  # TODO: 요청마다 종료 코드 0 또는 2를 먼저 적습니다
for event, guess in zip(events, my_guess):
    code, reason = guard(event)
    target = event['tool_input'].get('file_path') or event['tool_input'].get('command')
    print(f"{event['tool_name']:5} {target[:48]:48} → 종료 코드 {code}(예측 {guess}) {reason}")

## 실습 14-2. 나누면 토큰이 줄어드는가

교재의 절: "개념 확인 실습"

에이전트가 요청할 때마다 기본 묶음(`BASE`, 시스템 프롬프트와 도구 정의와 규칙 파일)과 앞 단계의 도구 결과(단계마다 `TOOL_OUTPUT`)를 모두 다시 보낸다고 가정합니다. 에이전트 하나가 `steps`번 요청할 때 보낸 입력 토큰의 합을 돌려주는 `agent_input()`을 작성합니다. 3단계 일과 24단계 일을 혼자 할 때와 서브에이전트 셋에 나눌 때 어느 쪽이 클지 먼저 `my_guess`에 적고, 결과가 일의 길이에 따라 왜 뒤집히는지 설명하면 끝납니다.

In [ ]:
BASE = 20_000        # 요청마다 다시 보내는 시스템 프롬프트, 도구 정의, 규칙 파일
TOOL_OUTPUT = 3_000  # 단계마다 대화에 더해지는 도구 결과
SUMMARY = 500        # 서브에이전트가 메인에 돌려주는 요약


def agent_input(steps):
    """에이전트 하나가 steps번 요청할 때 보낸 입력 토큰의 합. k번째 요청에는 BASE와 앞 k-1단계의 도구 결과가 모두 들어갑니다"""
    return 0  # TODO


def split_input(steps, workers):
    """일을 서브에이전트 workers개에 똑같이 나누고, 메인은 맡기는 요청과 요약을 합치는 요청 두 번만 보낼 때의 합"""
    main = BASE + (BASE + workers * SUMMARY)
    return workers * agent_input(steps // workers) + main


my_guess = {3: None, 24: None}  # TODO: 단계 수마다 '혼자' 또는 '나눔' 중 입력 토큰이 더 큰 쪽을 적습니다
for steps in [3, 24]:
    alone, split = agent_input(steps), split_input(steps, 3)
    print(f'{steps}단계: 혼자 {alone:,}토큰, 셋으로 나눔 {split:,}토큰 → 더 큰 쪽 {"혼자" if alone > split else "나눔"}(예측 {my_guess[steps]})')